# This is the search space

In [94]:
import sys
from pyprojroot import here
sys.path.insert(0, str(here()))

In [95]:
import yaml
import numpy as np
import optuna
import optunahub
from datasets import Dataset
from transformers import (
    RobertaForSequenceClassification,
    RobertaTokenizer,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
from sklearn.metrics import f1_score
from src.utils import get_pooled_df
from src.data import get_fold_from_disk, DECODED_LABELS

import matplotlib.pyplot as plt

In [96]:
#Let's load the base yaml for consistency
with open(here('configs/base.yaml'), 'r') as f: #I should probably make a dataclass here to deserialize but this works for now
    base_data = yaml.full_load(f)

print(data)

{'max_length': 256, 'cv_k': 5, 'cv_seed': 7, 'eval_strategy': 'epoch', 'save_strategy': 'epoch', 'load_best_model_at_end': True, 'metric_for_best_model': 'macro_f1', 'greater_is_better': True, 'early_stopping_patience': 3, 'fp16': True, 'seed': 42, 'report_to': 'none'}


In [98]:
full_df = get_pooled_df()
train_fold, val_fold = get_fold_from_disk(full_df, fold=4, k=5, seed=7)

tokenizer = RobertaTokenizer.from_pretrained("roberta-base")

def tokenize_fold(df):
    ds = Dataset.from_pandas(df, preserve_index=False)
    def _tokenize(examples):
        return tokenizer(
            examples["abstract_conclusion"],
            examples["press_release_conclusion"],
            truncation=True,
            padding="max_length",
            max_length=base_data['max_length'],
        )
    cols_to_remove = [c for c in ds.column_names if c != "exaggeration_label"]
    ds = ds.map(_tokenize, batched=True, remove_columns=cols_to_remove)
    ds = ds.rename_column("exaggeration_label", "labels")
    ds.set_format("torch")
    return ds

train_ds = tokenize_fold(train_fold)
val_ds = tokenize_fold(val_fold)
print(f"train={len(train_ds)} val={len(val_ds)}")

Map: 100%|██████████| 132/132 [00:00<00:00, 11417.07 examples/s]

train=531 val=132


In [99]:
#TPE visualizer https://hub.optuna.org/visualization/tpe_acquisition_visualizer/
module = optunahub.load_module(
    package="visualization/tpe_acquisition_visualizer",
)
tpe_acquisition_visualizer = module.TPEAcquisitionVisualizer()


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"macro_f1": f1_score(labels, preds, average="macro")}

In [100]:
#search space definition

def objective(trial):
    lr = trial.suggest_float("learning_rate", 3.2e-5, 4.1e-5, log=True)
    batch_size = trial.suggest_categorical("per_device_train_batch_size", [8, 16, 32])
    weight_decay = trial.suggest_float("weight_decay", 0.008, 0.2)
    warmup_ratio = trial.suggest_float("warmup_ratio", 0.002, 0.03)
    num_epochs = trial.suggest_int("num_train_epochs", 14,16)

    model = RobertaForSequenceClassification.from_pretrained("roberta-base", num_labels=3)
    #apply based_data fixed params here
    training_args = TrainingArguments(
        output_dir=str(here(f"results/optuna/trial-{trial.number}")),
        eval_strategy=base_data['eval_strategy'],
        save_strategy=base_data['save_strategy'],
        learning_rate=lr,
        per_device_train_batch_size=batch_size,
        weight_decay=weight_decay,
        warmup_ratio=warmup_ratio,
        num_train_epochs=num_epochs,
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=base_data['greater_is_better'],
        fp16=base_data['fp16'],
        seed=base_data['seed'],
        logging_steps=10,
        report_to=base_data['report_to'],
        save_total_limit=1,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
    )

    trainer.train()
    results = trainer.evaluate()
    return results["eval_macro_f1"]



In [101]:
#study definition
#prereques 
db_path = here("results/optuna/optuna_study.db")
storage_url = f"sqlite:///{db_path}"
print(storage_url) 


study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(
        n_startup_trials=n_startup_trials,
        seed=42,
    ),
    study_name="roberta-hyperparameter-searchspace_full_fold",
    storage=storage_url,
)


study.optimize(objective, n_trials=50, callbacks=[tpe_acquisition_visualizer])

#Best trails
print(f"\n{'='*60}")
print(f"  Best macro_f1: {study.best_trial.value:.4f}")
print(f"  Best params:   {study.best_trial.params}")
print(f"{'='*60}")



[I 2026-02-24 14:57:11,465] A new study created in RDB with name: roberta-hyperparameter-searchspace_full_fold


sqlite:////home/jubacochran/mids266-exaggeration/results/optuna/optuna_study.db


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 2653.89it/s, Materializing param=roberta.encoder.layer.11.output.dense.weight]              
RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consid

Epoch,Training Loss,Validation Loss,Macro F1
1,1.049306,0.942708,0.255452
2,0.864893,0.939864,0.255452
3,0.992050,0.912840,0.255452
4,0.837653,0.837958,0.255452


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.29s/it]
There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.la

...

[I 2026-02-24 15:32:58,525] Trial 48 finished with value: 0.5820363917924894 and parameters: {'learning_rate': 3.372234058770598e-05, 'per_device_train_batch_size': 8, 'weight_decay': 0.1764368858845944, 'warmup_ratio': 0.023635730317919348, 'num_train_epochs': 14}. Best is trial 6 with value: 0.5825232346971477.
Loading weights: 100%|██████████| 197/197 [00:00<00:00, 1395.05it/s, Materializing param=roberta.encoder.layer.11.output.dense.weight]              
RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSIN

Epoch,Training Loss,Validation Loss,Macro F1
1,1.039426,0.938462,0.255452
2,0.868466,0.968511,0.255452
3,0.997174,0.908175,0.255452
4,0.846287,0.865218,0.255452


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.16it/s]
There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.la

[I 2026-02-24 15:33:29,032] Trial 49 finished with value: 0.2554517133956386 and parameters: {'learning_rate': 3.368877472710883e-05, 'per_device_train_batch_size': 8, 'weight_decay': 0.19092037397835923, 'warmup_ratio': 0.02330891168192034, 'num_train_epochs': 15}. Best is trial 6 with value: 0.5825232346971477.



  Best macro_f1: 0.5825
  Best params:   {'learning_rate': 3.227389250956008e-05, 'per_device_train_batch_size': 8, 'weight_decay': 0.0678485266091669, 'warmup_ratio': 0.016561904592978703, 'num_train_epochs': 15}


In [103]:
#tpe acquistion 
viz_dir = here("results/optuna/tpe_plots_full_fold")
viz_dir.mkdir(parents=True, exist_ok=True)

param_names = ["learning_rate", "weight_decay", "warmup_ratio", "num_train_epochs","per_device_train_batch_size"]
for param_name in param_names:
    for trial in study.trials:
        if trial.number < n_startup_trials:
            continue
        try:
            fig = tpe_acquisition_visualizer.plot(study, trial.number, param_name)
            fig.savefig(viz_dir / f"{param_name}_trial{trial.number}.png")
            plt.close(fig)
        except Exception as e:
            print(f"Skipped {param_name} trial {trial.number}: {e}")

print(f"TPE acquisition plots saved to {viz_dir}")

Skipped per_device_train_batch_size trial 10: 'CategoricalDistribution' object has no attribute 'low'
Skipped per_device_train_batch_size trial 11: 'CategoricalDistribution' object has no attribute 'low'
Skipped per_device_train_batch_size trial 12: 'CategoricalDistribution' object has no attribute 'low'
Skipped per_device_train_batch_size trial 13: 'CategoricalDistribution' object has no attribute 'low'
Skipped per_device_train_batch_size trial 14: 'CategoricalDistribution' object has no attribute 'low'
Skipped per_device_train_batch_size trial 15: 'CategoricalDistribution' object has no attribute 'low'
Skipped per_device_train_batch_size trial 16: 'CategoricalDistribution' object has no attribute 'low'
Skipped per_device_train_batch_size trial 17: 'CategoricalDistribution' object has no attribute 'low'
Skipped per_device_train_batch_size trial 18: 'CategoricalDistribution' object has no attribute 'low'
Skipped per_device_train_batch_size trial 19: 'CategoricalDistribution' object has